# Métricas de clasificación: leer los errores con nombre propio

**Objetivo de aprendizaje:**
Al finalizar este notebook, el alumno podrá:
- Construir e interpretar una matriz de confusión (TP/TN/FP/FN)
- Calcular precisión, recall, F1-score y ROC-AUC y saber cuando usar cada uno
- Identificar por que accuracy engaña en datasets desbalanceados
- Decidir que métrica priorizar según el coste del error en un caso de negocio

**Conexión con el documento:**
Este notebook acompaña la sección `#### El coste del error` de `estructura_contenidos.md`.
La teoría de FP/FN ya esta explicada allí. Aqui la operacionalizamos con datos reales.

> Antes de seguir: si tuviera que elegir entre un modelo que no pierde ningun churn real
> pero lanza muchas falsas alarmas, y otro que lanza pocas alarmas pero se pierde el 40%%
> de los churns reales, cual elegiria para la empresa y por que?

<details>
<summary>Orientación para el instructor (desplegar tras la reflexión)</summary>

Una respuesta madura menciona: el coste relativo de perder un cliente vs el coste de
gestionar retenciones innecesarias, la capacidad del equipo de Customer Success para
absorber falsas alarmas, y que la elección depende del contexto y no de la matemática.

Si nadie responde, preguntar: 'Si Customer Success gestiona 50 contactos al mes,
que le cuesta mas: que 10 de esos 50 sean innecesarios, o que se escapen 5 churns?'

Señal de comprensión: el alumno distingue que recall alto conviene cuando el coste del
falso negativo es dominante, y que no existe una elección correcta universal.

</details>

## Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score, accuracy_score,
    roc_curve, auc, precision_recall_curve
)
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
IMG = 'images'
os.makedirs(IMG, exist_ok=True)

import sklearn, plotly
print('Librerias OK')
print(f'sklearn: {sklearn.__version__}  plotly: {plotly.__version__}')

Librerias OK
sklearn: 1.4.1.post1  plotly: 5.19.0


## 1. Ejemplo mínimo: matriz de confusión a mano

Antes de usar ninguna librería, construimos la matriz manualmente
para fijar la intuición de cada celda.

In [2]:
# 10 clientes: 1=churn, 0=no churn
y_real     = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0])
y_predicho = np.array([1, 0, 0, 1, 1, 0, 1, 0, 0, 0])

TP = int(np.sum((y_real == 1) & (y_predicho == 1)))
TN = int(np.sum((y_real == 0) & (y_predicho == 0)))
FP = int(np.sum((y_real == 0) & (y_predicho == 1)))
FN = int(np.sum((y_real == 1) & (y_predicho == 0)))

print('Matriz de confusion (manual):')
print(f'  TP (acerto churn):        {TP}')
print(f'  TN (acerto no-churn):     {TN}')
print(f'  FP (falsa alarma):        {FP}')
print(f'  FN (churn no detectado):  {FN}')

acc  = (TP + TN) / len(y_real)
prec = TP / (TP + FP) if (TP + FP) > 0 else 0
rec  = TP / (TP + FN) if (TP + FN) > 0 else 0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

print()
print(f'Accuracy:  {acc:.2f}  <- pct total de aciertos')
print(f'Precision: {prec:.2f}  <- de alertas lanzadas, cuantas eran validas')
print(f'Recall:    {rec:.2f}  <- de churns reales, cuantos detectamos')
print(f'F1-score:  {f1:.2f}  <- media armonica de precision y recall')

Matriz de confusion (manual):
  TP (acerto churn):        3
  TN (acerto no-churn):     4
  FP (falsa alarma):        1
  FN (churn no detectado):  2

Accuracy:  0.70  <- pct total de aciertos
Precision: 0.75  <- de alertas lanzadas, cuantas eran validas
Recall:    0.60  <- de churns reales, cuantos detectamos
F1-score:  0.67  <- media armonica de precision y recall


## 2. Ejemplo realista: dataset de churn 90/10

Simulamos 2000 clientes de la empresa con 90%% de renovación y 10%% de churn.
Mostramos primero el problema de accuracy en clases desbalanceadas.

In [3]:
# Dataset desbalanceado: 90%% no-churn, 10%% churn
X, y = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=5,
    weights=[0.90, 0.10],
    flip_y=0.02,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

n_churn    = int(np.sum(y == 1))
n_nochurn  = int(np.sum(y == 0))
pct_churn  = 100 * np.mean(y == 1)
pct_noch   = 100 * np.mean(y == 0)
print(f'Dataset total:  {len(y)} clientes')
print(f'  No churn (0): {n_nochurn} ({pct_noch:.0f}%%)')
print(f'  Churn    (1): {n_churn} ({pct_churn:.0f}%%)')
print(f'Test set:       {len(y_test)} clientes')

Dataset total:  2000 clientes
  No churn (0): 1780 (89%%)
  Churn    (1): 220 (11%%)
Test set:       600 clientes


In [4]:
# Modelo trivial: predice siempre 'no cancela'
y_trivial = np.zeros(len(y_test), dtype=int)
acc_trivial = accuracy_score(y_test, y_trivial)
churns_detectados = int(np.sum((y_test == 1) & (y_trivial == 1)))

print('Modelo trivial (predice siempre no-churn):')
print(f'  Accuracy:         {acc_trivial:.2f}  <- parece bueno')
print(f'  Churns detectados: {churns_detectados}  <- no detecta ninguno')
print('Conclusion: accuracy alta con recall=0. El peor modelo posible para negocio.')

Modelo trivial (predice siempre no-churn):
  Accuracy:         0.89  <- parece bueno
  Churns detectados: 0  <- no detecta ninguno
Conclusion: accuracy alta con recall=0. El peor modelo posible para negocio.


In [5]:
# Entrenar regresion logistica balanceada
lr = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

cm_trivial = confusion_matrix(y_test, y_trivial)
cm_lr      = confusion_matrix(y_test, y_pred_lr)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels_x = ['Pred: no churn', 'Pred: churn']
labels_y = ['Real: no churn', 'Real: churn']

for ax, cm, titulo in zip(
    axes,
    [cm_trivial, cm_lr],
    ['Modelo trivial (predice siempre 0)', 'Regresion Logistica (balanceada)']
):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels_x, yticklabels=labels_y,
                ax=ax, cbar=False, annot_kws={'size': 14})
    ax.set_title(titulo, fontsize=12, pad=10)
    ax.set_ylabel('Realidad', fontsize=11)
    ax.set_xlabel('Prediccion del modelo', fontsize=11)

plt.suptitle('Matriz de confusion: trivial vs logistica (90/10 churn)', fontsize=13)
plt.tight_layout()
plt.savefig('images/B02C_fig01.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B02C_fig01.png')

Guardado: B02C_fig01.png


In [6]:
modelos_dict = {
    'Trivial (siempre 0)': y_trivial,
    'Regresion Logistica': y_pred_lr,
}

filas = []
for nombre, yp in modelos_dict.items():
    filas.append({
        'Modelo': nombre,
        'Accuracy': round(accuracy_score(y_test, yp), 2),
        'Precision': round(precision_score(y_test, yp, zero_division=0), 2),
        'Recall': round(recall_score(y_test, yp, zero_division=0), 2),
        'F1': round(f1_score(y_test, yp, zero_division=0), 2),
    })

df_m = pd.DataFrame(filas)
print(df_m.to_string(index=False))
print()
print('El modelo trivial tiene 90%% accuracy pero 0%% recall.')
print('No detecta ningun churn real: el peor modelo posible para negocio.')

# Guardar tabla como imagen
fig, ax = plt.subplots(figsize=(9, 2.2))
ax.axis('off')
t = ax.table(cellText=df_m.values, colLabels=df_m.columns,
             cellLoc='center', loc='center')
t.auto_set_font_size(False)
t.set_fontsize(11)
t.scale(1.2, 1.6)
for j in range(len(df_m.columns)):
    t[0, j].set_facecolor('#2196F3')
    t[0, j].set_text_props(color='white', fontweight='bold')
for j in range(len(df_m.columns)):
    t[1, j].set_facecolor('#FFCDD2')  # rojo: trivial
for j in range(len(df_m.columns)):
    t[2, j].set_facecolor('#C8E6C9')  # verde: logistica
plt.title('Accuracy no cuenta la historia completa', fontsize=12, pad=8)
plt.tight_layout()
plt.savefig('images/B02C_fig02.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B02C_fig02.png')

             Modelo  Accuracy  Precision  Recall   F1
Trivial (siempre 0)      0.89       0.00    0.00 0.00
Regresion Logistica      0.74       0.24    0.65 0.36

El modelo trivial tiene 90%% accuracy pero 0%% recall.
No detecta ningun churn real: el peor modelo posible para negocio.
Guardado: B02C_fig02.png


## 3. Ejemplo avanzado: curvas ROC y precisión-recall

Comparamos la Regresión Logística con un Arbol de Decisión.
Las curvas ROC muestran el rendimiento a lo largo de todos los umbrales posibles.

In [7]:
# Segundo modelo: arbol de decision
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

prob_lr = lr.predict_proba(X_test)[:, 1]
prob_dt = dt.predict_proba(X_test)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(y_test, prob_lr)
fpr_dt, tpr_dt, _ = roc_curve(y_test, prob_dt)
auc_lr = auc(fpr_lr, tpr_lr)
auc_dt = auc(fpr_dt, tpr_dt)

prec_lr, rec_lr, _ = precision_recall_curve(y_test, prob_lr)
prec_dt, rec_dt, _ = precision_recall_curve(y_test, prob_dt)
ap_lr = float(np.trapz(prec_lr[::-1], rec_lr[::-1]))
ap_dt = float(np.trapz(prec_dt[::-1], rec_dt[::-1]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
ax = axes[0]
ax.plot(fpr_lr, tpr_lr, color='#1565C0', lw=2,
        label=f'Logistica (AUC={auc_lr:.3f})')
ax.plot(fpr_dt, tpr_dt, color='#E65100', lw=2,
        label=f'Arbol (AUC={auc_dt:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1,label='Aleatorio (AUC=0.50)')
ax.fill_between(fpr_lr, tpr_lr, alpha=0.08, color='#1565C0')
ax.set_xlabel('Tasa de falsos positivos (FPR)', fontsize=11)
ax.set_ylabel('Tasa de verdaderos positivos (Recall)', fontsize=11)
ax.set_title('Curva ROC - comparativa de modelos', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Precision-Recall
ax = axes[1]
ax.plot(rec_lr, prec_lr, color='#1565C0', lw=2,
        label=f'Logistica (AP={ap_lr:.3f})')
ax.plot(rec_dt, prec_dt, color='#E65100', lw=2,
        label=f'Arbol (AP={ap_dt:.3f})')
baseline = float(np.mean(y_test))
ax.axhline(baseline, color='k', linestyle='--', lw=1,
           label=f'Baseline ({baseline:.2f})')
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Curva Precision-Recall - trade-off visible', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('images/B02C_fig03.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Logistica:  AUC ROC={auc_lr:.3f} | AP={ap_lr:.3f}')
print(f'Arbol Dec.: AUC ROC={auc_dt:.3f} | AP={ap_dt:.3f}')
print('Guardado: B02C_fig03.png')

Logistica:  AUC ROC=0.771 | AP=0.227
Arbol Dec.: AUC ROC=0.831 | AP=0.543
Guardado: B02C_fig03.png


In [8]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

datos_celdas = [
    (2.5, 7.2, '#C8E6C9', '#1B5E20', 'TP  Verdadero Positivo',
     'Predijo churn - SI cancelo', 'Accion justificada [OK]'),
    (7.5, 7.2, '#FFCDD2', '#B71C1C', 'FN  Falso Negativo',
     'Predijo ok - PERO cancelo', 'Cliente perdido [X] Peor error'),
    (2.5, 2.8, '#FFF9C4', '#F57F17', 'FP  Falso Positivo',
     'Predijo churn - NO cancelo', 'Recurso desperdiciado [!]'),
    (7.5, 2.8, '#E3F2FD', '#0D47A1', 'TN  Verdadero Negativo',
     'Predijo ok - Y no cancelo', 'Sin accion necesaria [OK]'),
]

for (x, y, cfondo, ctexto, titulo, desc, impacto) in datos_celdas:
    rect = mpatches.FancyBboxPatch(
        (x-2.3, y-2.0), 4.6, 4.0,
        boxstyle='round,pad=0.1', linewidth=1.5,
        edgecolor=ctexto, facecolor=cfondo
    )
    ax.add_patch(rect)
    ax.text(x, y+1.2, titulo, ha='center', va='center',
            fontsize=10, fontweight='bold', color=ctexto)
    ax.text(x, y-0.0, desc, ha='center', va='center',
            fontsize=9, color='#333333')
    ax.text(x, y-1.1, impacto, ha='center', va='center',
            fontsize=9, color=ctexto, style='italic')

ax.text(5, 9.5, 'Matriz de confusion - lectura orientada a negocio (churn)',
        ha='center', va='center', fontsize=13, fontweight='bold')
ax.text(2.5, 5.05, 'MODELO: CHURN', ha='center', va='center', fontsize=9,
        color='#555', bbox=dict(boxstyle='round', facecolor='#ECEFF1'))
ax.text(7.5, 5.05, 'MODELO: NO CHURN', ha='center', va='center', fontsize=9,
        color='#555', bbox=dict(boxstyle='round', facecolor='#ECEFF1'))
ax.text(0.2, 7.2, 'CANCELO\n(real)', ha='center', va='center',
        fontsize=9, color='#555', rotation=90)
ax.text(0.2, 2.8, 'NO CANCELO\n(real)', ha='center', va='center',
        fontsize=9, color='#555', rotation=90)

plt.tight_layout()
plt.savefig('images/B02C_fig04.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B02C_fig04.png')

Guardado: B02C_fig04.png


## 4. Visualización interactiva: slider de umbral de decisión

In [9]:
umbrales = np.linspace(0.01, 0.99, 100)
precs, recs, f1s, accs = [], [], [], []

for thr in umbrales:
    yp = (prob_lr >= thr).astype(int)
    precs.append(precision_score(y_test, yp, zero_division=0))
    recs.append(recall_score(y_test, yp, zero_division=0))
    f1s.append(f1_score(y_test, yp, zero_division=0))
    accs.append(accuracy_score(y_test, yp))

fig_p = go.Figure()
fig_p.add_trace(go.Scatter(x=umbrales, y=precs, name='Precision',
                           line=dict(color='#1565C0', width=2)))
fig_p.add_trace(go.Scatter(x=umbrales, y=recs, name='Recall',
                           line=dict(color='#E65100', width=2)))
fig_p.add_trace(go.Scatter(x=umbrales, y=f1s, name='F1-score',
                           line=dict(color='#2E7D32', width=2, dash='dash')))
fig_p.add_trace(go.Scatter(x=umbrales, y=accs, name='Accuracy',
                           line=dict(color='#9E9E9E', width=1.5, dash='dot')))
fig_p.add_vline(x=0.5, line_dash='dash', line_color='#555',
                annotation_text='umbral=0.5 (default)')
fig_p.update_layout(
    title='Precision, Recall y F1 segun el umbral de decision',
    xaxis_title='Umbral de probabilidad',
    yaxis_title='Metrica',
    yaxis=dict(range=[0, 1.05]),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=450,
    template='plotly_white'
)

html_path = 'images/B02C_fig05.html'
fig_p.write_html(html_path)
print(f'Guardado HTML: {html_path}')

try:
    fig_p.write_image('images/B02C_fig05.png', width=900, height=450)
    print('Guardado PNG: B02C_fig05.png')
except Exception as e:
    print(f'PNG no disponible (kaleido): {e}')

fig_p.show()

Guardado HTML: images/B02C_fig05.html
PNG no disponible (kaleido): 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [10]:
umbrales_key = [0.2, 0.3, 0.5, 0.7, 0.8]
filas_thr = []
for thr in umbrales_key:
    yp = (prob_lr >= thr).astype(int)
    tp_k = int(np.sum((y_test == 1) & (yp == 1)))
    fn_k = int(np.sum((y_test == 1) & (yp == 0)))
    filas_thr.append({
        'Umbral': thr,
        'Precision': round(precision_score(y_test, yp, zero_division=0), 2),
        'Recall': round(recall_score(y_test, yp, zero_division=0), 2),
        'F1': round(f1_score(y_test, yp, zero_division=0), 2),
        'Accuracy': round(accuracy_score(y_test, yp), 2),
        'TP detect.': tp_k,
        'FN perdidos': fn_k,
    })

df_thr = pd.DataFrame(filas_thr)
print(df_thr.to_string(index=False))
print()
print('Umbral 0.3: mas churns detectados, mas falsas alarmas.')
print('Umbral 0.7: menos falsas alarmas, pero se escapan mas churns.')

fig, ax = plt.subplots(figsize=(12, 2.8))
ax.axis('off')
t = ax.table(cellText=df_thr.values, colLabels=df_thr.columns,
             cellLoc='center', loc='center')
t.auto_set_font_size(False)
t.set_fontsize(10)
t.scale(1.1, 1.6)
for j in range(len(df_thr.columns)):
    t[0, j].set_facecolor('#1565C0')
    t[0, j].set_text_props(color='white', fontweight='bold')
for j in range(len(df_thr.columns)):
    t[2, j].set_facecolor('#E8F5E9')  # umbral 0.3
plt.title('Metricas en umbrales clave (verde: umbral sugerido para churn)', fontsize=11)
plt.tight_layout()
plt.savefig('images/B02C_fig06.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B02C_fig06.png')

 Umbral  Precision  Recall   F1  Accuracy  TP detect.  FN perdidos
    0.2       0.18    0.92 0.30      0.52          61            5
    0.3       0.20    0.83 0.32      0.61          55           11
    0.5       0.24    0.65 0.36      0.74          43           23
    0.7       0.25    0.38 0.30      0.81          25           41
    0.8       0.23    0.21 0.22      0.83          14           52

Umbral 0.3: mas churns detectados, mas falsas alarmas.
Umbral 0.7: menos falsas alarmas, pero se escapan mas churns.
Guardado: B02C_fig06.png


## 5. Ejercicio técnico

**Enunciado:**
El equipo de BI entrega los resultados de un modelo de clasificación de tickets urgentes.
De 500 tickets en el periodo de prueba:
- 80 tickets eran realmente urgentes
- El modelo marcó 60 como urgentes
- De esos 60, 45 eran realmente urgentes

Calcula TP, TN, FP, FN, Accuracy, Precisión, Recall y F1 a mano.
Verifica con sklearn.

In [11]:
# Datos del ejercicio - no modificar
# 500 tickets: 80 urgentes reales, 60 predichos urgentes, 45 correctos
np.random.seed(99)

# TP=45, FP=15, FN=35, TN=405
y_real_ej = np.array([1]*80 + [0]*420)
y_pred_ej = np.array([1]*45 + [0]*35 + [1]*15 + [0]*405)

print(f'Total tickets:   {len(y_real_ej)}')
print(f'Urgentes reales: {int(np.sum(y_real_ej))}')
print(f'Pred. urgentes:  {int(np.sum(y_pred_ej))}')

Total tickets:   500
Urgentes reales: 80
Pred. urgentes:  60


In [12]:
# TODO: calcula las metricas a mano
# ────────────────────────────────────────────────────────────────────
TP_calc = None  # np.sum((y_real_ej == 1) & (y_pred_ej == 1))
TN_calc = None
FP_calc = None
FN_calc = None

acc_calc  = None  # (TP + TN) / total
prec_calc = None  # TP / (TP + FP)
rec_calc  = None  # TP / (TP + FN)
f1_calc   = None  # 2 * prec * rec / (prec + rec)

print('Tus calculos:')
print(f'  Accuracy:  {acc_calc}')
print(f'  Precision: {prec_calc}')
print(f'  Recall:    {rec_calc}')
print(f'  F1-score:  {f1_calc}')

# Verifica con sklearn
print()
print('Verificacion sklearn:')
print(classification_report(y_real_ej, y_pred_ej,
                            target_names=['Normal', 'Urgente']))

Tus calculos:
  Accuracy:  None
  Precision: None
  Recall:    None
  F1-score:  None

Verificacion sklearn:
              precision    recall  f1-score   support

      Normal       0.92      0.96      0.94       420
     Urgente       0.75      0.56      0.64        80

    accuracy                           0.90       500
   macro avg       0.84      0.76      0.79       500
weighted avg       0.89      0.90      0.89       500



**Pista:** empieza calculando TP con `np.sum((y_real_ej == 1) & (y_pred_ej == 1))`.
Con TP, FP, FN, TN el resto son fracciones simples.

In [13]:
# SOLUCION (ejecutar despues de intentarlo)
TP_s = int(np.sum((y_real_ej == 1) & (y_pred_ej == 1)))
TN_s = int(np.sum((y_real_ej == 0) & (y_pred_ej == 0)))
FP_s = int(np.sum((y_real_ej == 0) & (y_pred_ej == 1)))
FN_s = int(np.sum((y_real_ej == 1) & (y_pred_ej == 0)))

acc_s  = (TP_s + TN_s) / len(y_real_ej)
prec_s = TP_s / (TP_s + FP_s)
rec_s  = TP_s / (TP_s + FN_s)
f1_s   = 2 * prec_s * rec_s / (prec_s + rec_s)

print('=== Solucion ===')
print(f'TP={TP_s}, TN={TN_s}, FP={FP_s}, FN={FN_s}')
print(f'Accuracy:  {acc_s:.3f}')
print(f'Precision: {prec_s:.3f}  <- de alertas lanzadas, {prec_s*100:.0f}%% eran validas')
print(f'Recall:    {rec_s:.3f}  <- detectamos el {rec_s*100:.0f}%% de los urgentes reales')
print(f'F1-score:  {f1_s:.3f}')
print()
print('Interpretacion:')
print(f'  {FN_s} tickets urgentes no se priorizaron (FN).')
print(f'  {FP_s} tickets normales se procesaron como urgentes (FP).')

=== Solucion ===
TP=45, TN=405, FP=15, FN=35
Accuracy:  0.900
Precision: 0.750  <- de alertas lanzadas, 75%% eran validas
Recall:    0.562  <- detectamos el 56%% de los urgentes reales
F1-score:  0.643

Interpretacion:
  35 tickets urgentes no se priorizaron (FN).
  15 tickets normales se procesaron como urgentes (FP).


## 6. Ejercicio de decisión

**Caso:**
El equipo de Customer Success va a desplegar el modelo de churn. Dos opciones:

| Modo | Umbral | Precisión | Recall | F1 | Descripción |
|---|---|---|---|---|---|
| A | 0.3 | 0.41 | 0.82 | 0.55 | Detecta 82 de cada 100 churns; doble de alertas innecesarias |
| B | 0.6 | 0.71 | 0.55 | 0.62 | Detecta 55 de cada 100 churns; casi todas las alertas son válidas |

**Contexto operativo:**
- Customer Success puede gestionar 80 contactos de retención al mes
- Hay aproximadamente 30 churns reales al mes
- Perder un cliente: 3.000 euros de coste
- Gestión de retención innecesaria: 50 euros (tiempo del equipo)

**Pregunta:**
Que modo elegirias? Maximizarias recall o precisión?
Cambia tu decisión si Customer Success pudiera contratar un agente adicional?

**Tu razonamiento:**

(escribe aqui tu argumentación)

---

**Criterios de evaluación (para el instructor):**

Una respuesta sólida incluye:
1. Cálculo del coste: Modo A pierde ~5 churns x 3.000 = 15.000 euros de pérdida;
   Modo B pierde ~14 churns x 3.000 = 42.000 euros. Modo A gana aunque tiene mas FP.
2. Reconocer que la capacidad del equipo (80 contactos/mes) es una restricción operativa,
   no del modelo. Contratar un agente cambia el cálculo.
3. Identificar que en churn de alto valor unitario, recall suele dominar.
4. Concluir que el umbral es una decisión de negocio, no técnica.

## 7. Assets generados

| Fichero | Contenido |
|---|---|
| `images/B02C_fig01.png` | Matrices de confusión: trivial vs logística |
| `images/B02C_fig02.png` | Tabla comparativa de métricas (accuracy vs F1) |
| `images/B02C_fig03.png` | Curvas ROC y Precisión-Recall para dos modelos |
| `images/B02C_fig04.png` | Esquema visual: TP/TN/FP/FN en contexto de negocio |
| `images/B02C_fig05.html` | Plotly interactivo: precisión/recall/F1 por umbral |
| `images/B02C_fig06.png` | Tabla de métricas en umbrales clave |

**Dependencias:** numpy, pandas, matplotlib, seaborn, scikit-learn, plotly